# Stage 3: alignment against engine reference checks (DPO)

The distinctive stage. Three steps, all driven by the CLI:

1. **Sample candidates**: the SFT model answers every align-split position once per configured temperature, so the same position gets careful and careless attempts.
2. **Verify and score**: the claim parser reads each candidate, the verifier checks every claim against the stored Stockfish facts, and the JAX scorer ranks all candidates in one compiled batch. Preference pairs come out of engine reference checks, not human ratings.
3. **DPO**: trl trains the adapter to prefer the factually clean answers.

- **Runtime type**: **T4 GPU**.
- **Time**: roughly one to two hours (generation dominates).


In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/ariyan-tipu/opening-theory-engine.git"

# A plain install rather than -e: editable installs only register when the
# interpreter starts, and a live Colab kernel has already started.
!rm -rf /content/opening-theory-engine
!git clone --depth 1 {REPO_URL} /content/opening-theory-engine
%pip install -q "/content/opening-theory-engine[train,gpu]"
print("install done")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/opening-theory-engine

CONFIG = "/content/opening-theory-engine/configs/colab-t4.yaml"

In [ ]:
# Optional: live loss curves on wandb.ai (free account). This only surfaces the
# key into the environment; the training subprocess does the actual wandb login,
# so nothing heavy is imported into this kernel.
import os

key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata

        key = userdata.get("WANDB_API_KEY")
    except Exception:  # noqa: BLE001  (not on Colab, or no secret granted)
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key
    print("W&B key found: this stage will log online.")
else:
    print("No WANDB_API_KEY found: this stage will log offline.")
    print("For live dashboards, add WANDB_API_KEY as a Colab secret and re-run.")

In [ ]:
# Step 1: sample candidate explanations at every configured temperature.
!ote generate candidates --config {CONFIG}

In [ ]:
# Step 2: verify claims, score with JAX, keep pairs the evidence supports.
!ote align pairs --config {CONFIG}
print(open("data/work/pair_stats.json", encoding="utf-8").read())

In [ ]:
# Step 3: DPO, starting from the SFT adapter.
!ote train dpo --config {CONFIG}
print(open("artefacts/dpo/dpo_metrics.json", encoding="utf-8").read())

Both adapters (`artefacts/sft/adapter`, `artefacts/dpo/adapter`) are now on your Drive. Finish with `04_evaluate_and_demo.ipynb`.
